# Note : not aviable explicit cashe on free trail

### Explicit (Manual) Caching
* **Definition:** Explicit caching mein aap khud `client.caches.create()` ke zariye bhari context (jaise PDF, docs, ya lamba prompt) aur TTL (`ttl`) set karke cache banate hain, aur phir har request mein `cached_content=cache.name` pass karte hain.
* **Faida:** Is se **guaranteed cache reuse** hota hai, jisse latency kam hoti hai aur repeat input tokens ki cost mein clear saving milti hai.
* **Lifecycle & Cleanup:** Cache ki aik expiry (`ttl`) hoti hai, aur kaam khatam hone ke baad `client.caches.delete(name=cache.name)` se delete karna behtar hota hai taake extra storage charges na lagain.

In [ ]:
import os
from google import genai
from google.genai import types
import httpx
import time
from dotenv import load_dotenv

load_dotenv()  # Load environment variables from .env file

client = genai.Client()
MODEL_ID = "gemini-3.5-flash"
PDF_PATH = r"C:\Users\abubakar\Desktop\agentic_notebook\00_data\parsing_data\elon_musk.pdf"
PROMPT = "Provide a concise, well-structured summary of the key concepts in this document."

# Baseline Call (Without Cache + Token Usage)

In [ ]:
client = genai.Client()

# 2. Read the local PDF file bytes directly
with open(PDF_PATH, "rb") as f:
    doc_data = f.read()

prompt = "Summarize this document"
start= time.time()
response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=[
        types.Part.from_bytes(
            data=doc_data,
            mime_type='application/pdf',
        ),
        prompt
    ]
)
end=time.time()
print(response.text)
total_time_difference= end-start
print(total_time_difference)

In [ ]:
# 1. Extract token usage from the response
print(f"Cached tokens: {response.usage_metadata.cached_content_token_count}")
print(f"Input tokens:  {response.usage_metadata.prompt_token_count}")
print(f"Output tokens: {response.usage_metadata.candidates_token_count}")
print(f"Total tokens:  {response.usage_metadata.total_token_count}")

# Upload PDF & Create Explicit Cache

In [ ]:
# 2. Upload your PDF using the Files API

print(f"Uploading {PDF_PATH}...")
document = client.files.upload(
    file=PDF_PATH, 
    config={'mime_type': 'application/pdf'}
)

# Wait for the file to finish processing
while document.state.name == 'PROCESSING':
    print("Processing file...")
    time.sleep(2)
    document = client.files.get(name=document.name)

print(f"File ready: {document.uri}")

# 3. Create the Cache Manually (Explicit Caching)
system_instruction = "You are an expert at analyzing documents."

print("Creating manual context cache...")
cache = client.caches.create(
    model=MODEL_ID,
    config=types.CreateCachedContentConfig(
        system_instruction=system_instruction,
        contents=[document],
        ttl="360s",
        display_name="jupyter_pdf_cache"
    )
)

print(f"Cache created successfully! Cache Name: {cache.name}")

In [ ]:
# List Active Caches(META DATA OF CACHED DATA) 
for c in client.caches.list():
    print(f"Active Cache: {c.name} | Display Name: {c.display_name}")

#### First Cached Query + Token Usage

In [ ]:
# 4. Generate content using the cached tokens
start = time.time()
response = client.models.generate_content(
    model=MODEL_ID,
    contents="Please provide a detailed summary of this document.",
    config=types.GenerateContentConfig(cached_content=cache.name)
)
elapsed = time.time() - start

print(f"Time taken (Cached Query 1): {elapsed:.2f}s")
print(f"Cached tokens: {response.usage_metadata.cached_content_token_count}")
print(f"Input tokens:  {response.usage_metadata.prompt_token_count}")
print(f"Output tokens: {response.usage_metadata.candidates_token_count}")
print(f"Total tokens:  {response.usage_metadata.total_token_count}\n")
print(response.text[:400] + "...")

## Second Cached Query + Token Usage

In [ ]:
start = time.time()
response = client.models.generate_content(
    model=MODEL_ID,
    contents="Explain SOLAR SPECTRUM – ELECTROMAGNETIC RADIATION – RADIENT ENERGY",
    config=types.GenerateContentConfig(cached_content=cache.name)
)
elapsed = time.time() - start

print(f"Time taken (Cached Query 2): {elapsed:.2f}s")
print(f"Cached tokens: {response.usage_metadata.cached_content_token_count}")
print(f"Input tokens:  {response.usage_metadata.prompt_token_count}")
print(f"Output tokens: {response.usage_metadata.candidates_token_count}")
print(f"Total tokens:  {response.usage_metadata.total_token_count}\n")
print(response.text[:400] + "...")

## Delete Cache & Verify Cleanup

In [ ]:
# Loop through and delete every active cache
for cache in client.caches.list():
    client.caches.delete(name=cache.name)
    print(f"Successfully deleted: {cache.name}")

In [ ]:
for cache in client.caches.list():
  print(cache)